# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshiniChebrolu/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
### Unit of analysis + time window

- **Unit of analysis:** One content item for one client on one report date.
- **Grain:** `report_date + client_hash_id + content_hash_id`
- **Decision window:** Use information available up to the decision point to predict the subsequent performance direction.
- **Mid-panel month used for verification:** March 2026.
- **Observed March 2026 window:** 2026-03-01 to 2026-03-31.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 setup
!pip -q install duckdb huggingface_hub pandas

import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

print("Setup complete.")

Setup complete.


In [32]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully." if HF_TOKEN else "HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [33]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [34]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully." if HF_TOKEN else "HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [35]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully." if HF_TOKEN else "HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [36]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", bool(HF_TOKEN))

Token loaded: True


In [37]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded:", march_file)

Downloaded: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [38]:
import duckdb

con = duckdb.connect()

df = con.execute(f"""
    SELECT *
    FROM read_parquet('{march_file}')
    LIMIT 5
""").fetchdf()

df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [39]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
    FROM read_parquet('{march_file}')
""").fetchdf()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,9841378


In [40]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{march_file}')
""").fetchdf()

result

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Fields

##### Features

- `gsc_impressions` — knowable at decision moment because it is observed historical GSC performance.
- `gsc_clicks` — knowable at decision moment because it is observed historical GSC performance.
- `gsc_avg_position` — knowable at decision moment because it is observed historical GSC performance.
- `ga4_sessions` — knowable at decision moment because it is observed historical GA4 performance when available.
- `scroll_events` — knowable at decision moment because it is observed historical engagement performance.
Each feature is based on observed historical performance available before the prediction window.

#### Label

- `performance_direction` — future performance direction based on the change in monthly `gsc_impressions` from March 2026 to April 2026:
  - `grow` — April impressions > March impressions
  - `decline` — April impressions < March impressions
  - `recover` — April impressions = March impressions
#### Context

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`

#### Excluded

- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`

**Why excluded:** These fields are sparse and are not required for the core growth/recovery/momentum prediction contract.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [42]:
columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{march_file}')
""").fetchdf()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{march_file}')
""").fetchdf()

result

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [44]:
future_check = con.execute(f"""
    SELECT
        report_date,
        COUNT(*) AS rows
    FROM read_parquet('{march_file}')
    GROUP BY report_date
    ORDER BY report_date
    LIMIT 5
""").fetchdf()

future_check

,report_date,rows
0,2026-03-01,275874
1,2026-03-02,276269
2,2026-03-03,311676
3,2026-03-04,311675
4,2026-03-05,311676


In [45]:
future_dates = con.execute("""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        '/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=*/data_0.parquet'
    )
""").fetchdf()

future_dates

,min_date,max_date
0,2026-03-01,2026-06-30


In [46]:
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

may_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-05/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

june_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-06/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("April, May, June downloaded.")

April, May, June downloaded.


In [47]:
result = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet('{march_file}')
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{april_file}')
    GROUP BY client_hash_id, content_hash_id
),
labels AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN april_impressions > march_impressions THEN 'grow'
            WHEN april_impressions < march_impressions THEN 'decline'
            ELSE 'recover'
        END AS performance_direction
    FROM march
    JOIN april USING (client_hash_id, content_hash_id)
)
SELECT
    performance_direction,
    COUNT(*) AS count
FROM labels
GROUP BY performance_direction
ORDER BY performance_direction
""").fetchdf()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,performance_direction,count
0,decline,111967
1,grow,79514
2,recover,139955


In [48]:
from sklearn.metrics import accuracy_score

# Deliberately use the label itself as a feature
y = result["performance_direction"]

leakage_prediction = y.copy()

score = accuracy_score(y, leakage_prediction)

print("Leakage feature accuracy:", score)

Leakage feature accuracy: 1.0


In [49]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Create March-level features
features = march.groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    ga4_sessions=("ga4_sessions", "sum"),
    scroll_events=("scroll_events", "sum")
).reset_index()

# Create March → April label
april = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{april_file}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

features = features.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

features["performance_direction"] = features.apply(
    lambda x: "grow" if x["april_impressions"] > x["gsc_impressions"]
    else "decline" if x["april_impressions"] < x["gsc_impressions"]
    else "recover",
    axis=1
)

X = features[
    ["gsc_impressions", "gsc_clicks", "gsc_avg_position",
     "ga4_sessions", "scroll_events"]
]

y = features["performance_direction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest accuracy:", accuracy_score(y_test, pred))

Honest accuracy: 0.7555364470190683


### Leakage check

A deliberately label-derived feature produced a perfect accuracy of 1.00, demonstrating target leakage.

After removing the leakage feature and using only information available at the decision moment, the model achieved an honest test accuracy of 75.55%.

The honest score is retained for evaluation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
### Data limits

- The data is unbalanced across clients, content items, and dates, so observed performance may not represent every content item equally.
- Some early records are GSC-only because GA4 data may not be available; missing GA4 values should not automatically be interpreted as zero traffic.
- The daily performance data ends at 2026-06-30, so the most recent days outside this window cannot be evaluated.
- Historical performance can support directional decision-making, but it cannot establish that a feature caused a change in performance.

In [50]:
result = con.execute(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS total_rows
    FROM read_parquet([
        '{march_file}',
        '{april_file}',
        '{may_file}',
        '{june_file}'
    ])
""").fetchdf()

result

,min_date,max_date,total_rows
0,2026-03-01,2026-06-30,43647556


In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [52]:
result = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet('{march_file}')
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{april_file}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN april_impressions > march_impressions THEN 'grow'
        WHEN april_impressions < march_impressions THEN 'decline'
        ELSE 'recover'
    END AS performance_direction,
    COUNT(*) AS count
FROM march
JOIN april USING (client_hash_id, content_hash_id)
GROUP BY performance_direction
ORDER BY performance_direction
""").fetchdf()

result

,performance_direction,count
0,decline,111967
1,grow,79514
2,recover,139955


In [53]:
print("Data limits documented:")
print("1. Unbalanced data across clients, content items, and dates.")
print("2. GA4 may be unavailable for some records.")
print("3. Daily data ends at 2026-06-30.")
print("4. Historical features do not establish causality.")

Data limits documented:
1. Unbalanced data across clients, content items, and dates.
2. GA4 may be unavailable for some records.
3. Daily data ends at 2026-06-30.
4. Historical features do not establish causality.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.